%md
## Transform Population By Age data by performing the transformations below
####-----------------------------------------------------------------------
1. Split the country code & age group
2. Exclude all data other than 2019
3. Remove non numeric data from percentage
4. Pivot the data by age group
5. Join to dim_country to get the country, 3 digit country code and the total population.

####-----------------------------------------------------------------------

In [0]:
from pyspark.sql.functions import regexp_replace, split, col , regexp_extract, expr

### Read the population data and get required columns in dataframe

In [0]:
df_raw_population = spark.read.csv("abfss://datalake@adlstoolsjuly.dfs.core.windows.net/Covid19_project/population_raw/population", sep=r'\t', header=True, inferSchema = True)



In [0]:
df_raw_population_modified = (
    df_raw_population
    .withColumn("age_group",regexp_replace(split(col("indic_de,geo\\time"), ",")[0],        "^PC_",""))
    .withColumn(
        "country_code", regexp_extract(
            split(col("indic_de,geo\\time"), ",")[1],
            "^([A-Za-z]{2})",1)
    )
    .withColumn(
        "percentage_2019",
        expr("""
            try_cast(
                regexp_replace(trim(`2019 `), '[a-zA-Z]', '')
                AS decimal(4,2)
            )
        """)
    )
    .select("country_code","age_group","percentage_2019" )
)

%md
### Pivot the data by age group

In [0]:
from pyspark.sql.functions import col

age_groups = [
    "Y0_14",
    "Y15_24",
    "Y25_49",
    "Y50_64",
    "Y65_79",
    "Y80_MAX"
]

df_population_pivot = (
    df_raw_population_modified
    .groupBy("country_code")
    .pivot("age_group", age_groups)
    .sum("percentage_2019")
    .select(
        col("country_code"),
        col("Y0_14").alias("age_group_0_14"),
        col("Y15_24").alias("age_group_15_24"),
        col("Y25_49").alias("age_group_25_49"),
        col("Y50_64").alias("age_group_50_64"),
        col("Y65_79").alias("age_group_65_79"),
        col("Y80_MAX").alias("age_group_80_max")
    )
    .orderBy("country_code")
)

#display(df_population_pivot)


country_code,age_group_0_14,age_group_15_24,age_group_25_49,age_group_50_64,age_group_65_79,age_group_80_max
AD,13.90,10.60,39.40,22.50,10.20,3.40
AL,17.20,15.50,33.00,20.20,11.40,2.70
AM,20.20,11.80,36.90,19.10,9.00,3.00
AT,14.40,10.90,34.00,21.70,13.80,5.00
AZ,22.40,14.10,39.10,17.60,5.30,1.50
BE,16.90,11.40,32.70,20.10,13.30,5.60
BG,14.40,8.90,35.00,20.40,16.50,4.80
BY,16.90,9.90,36.60,21.30,11.30,3.90
CH,15.00,10.60,35.00,20.90,13.30,5.20
CY,16.10,12.80,37.10,17.90,12.50,3.70


### Read the country lookup

In [0]:
# Create a data frame for the country lookup
df_dim_country = spark.read.csv("abfss://datalake@adlstoolsjuly.dfs.core.windows.net/Covid19_project/lookup/country_lookup", sep=r',', header=True).select("country","country_code_2_digit","country_code_3_digit","population")


### Join population data with country lookup

In [0]:
df_population_joined = df_population_pivot.join(df_dim_country, df_population_pivot.country_code == df_dim_country.country_code_2_digit, how="inner")

In [0]:
df_population_processed = (
    df_population_joined
    .select(
                col("country"),
                 col("country_code_2_digit"),
                 col("country_code_3_digit"),
                 col("population"),
                 col("age_group_0_14"),
                 col("age_group_15_24"),
                 col("age_group_25_49"),
                 col("age_group_50_64"),
                 col("age_group_65_79"),
                 col("age_group_80_max")
                 
    ).orderBy("country")
)



###Write output to the processed folder in CSV file format

In [0]:
output_path = "abfss://datalake@adlstoolsjuly.dfs.core.windows.net/Covid19_project/processed/population"

df_population_processed.write.mode("overwrite").option("header", "true").option("delimiter", ",").csv(output_path)